# D6_02 부스팅: GBM과 LightGBM

> **오늘 질문:** 앞 나무의 실수를 다음 나무가 고치는 방식은 얼마나 강할까? 설정에 얼마나 민감할까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 3일차 Data Mart 📋

In [ ]:
%matplotlib inline
import time
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from lightgbm import LGBMClassifier

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]

cut = int(len(df) * 0.8)
X_train, X_test = df[features].iloc[:cut], df[features].iloc[cut:]
y_train, y_test = df["failure"].iloc[:cut], df["failure"].iloc[cut:]

## 1. Gradient Boosting (scikit-learn) ✍️

학습 시간도 함께 잽니다.

> 변수 이름: `gbm`

In [ ]:
# 힌트: GradientBoostingClassifier(random_state=42), time.time()

## 2. LightGBM ✍️

같은 Gradient Boosting을 훨씬 빠르게 만든 라이브러리입니다. `verbose=-1`은 학습 로그를 끕니다.

> 변수 이름: `lgbm`

In [ ]:
# 힌트: LGBMClassifier(random_state=42, verbose=-1)

## 3. learning_rate와 n_estimators 바꿔 보기 📋

'한 번에 고치는 정도(learning_rate)'와 '나무 개수(n_estimators)'를 바꿔 가며 Train·Test F1을 봅니다.

In [ ]:
rows = []
for lr in [0.3, 0.1, 0.03]:
    for n in [50, 200, 500]:
        m = LGBMClassifier(learning_rate=lr, n_estimators=n, random_state=42, verbose=-1).fit(X_train, y_train)
        rows.append({"learning_rate": lr, "n_estimators": n,
                     "Train F1": f1_score(y_train, m.predict(X_train)),
                     "Test F1": f1_score(y_test, m.predict(X_test))})
grid = pd.DataFrame(rows)
grid.pivot(index="learning_rate", columns="n_estimators", values="Test F1").round(3)

## 4. 랜덤포레스트와 비교 ✍️

배깅 대표(랜덤포레스트)와 부스팅 대표(LightGBM)를 정밀도·재현율까지 나란히 봅니다.

## 정리 💬

**Q1.** GBM과 LightGBM의 성능과 학습 시간을 비교해 보세요.

> 답:


**Q2.** learning_rate·n_estimators 표에서 Test F1이 크게 바뀌나요? 랜덤포레스트와 비교해 이 데이터에서 무엇을 고르겠습니까?

> 답:
